## Phase 2 — Representative Modeling Sample

Create a computationally efficient, population-representative subset of the ACS Individuals dataset using `PWGTP` survey weights.  
The goal is to preserve national demographic and socioeconomic distributions while reducing data volume for machine learning workflows.

Key steps:
- Perform weighted sampling using `PWGTP`
- Validate distribution alignment against the full dataset
- Export the modeling-ready dataset for downstream clustering and ML

In [4]:
# Import libraries and set global configuration

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [5]:
# Load the structural Individuals dataset (output of Notebook 02)

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"

# Structural dataset path
DATA_PATH = INTERIM_DIR / "acs_pums_5y"
FILE_NAME = "acs_structural_person_v1_1.parquet"

STRUCT_PATH = DATA_PATH / FILE_NAME

if not STRUCT_PATH.exists():
    raise FileNotFoundError(
        "Structural dataset not found. Run notebook 02_acs_structural_features first."
    )

df_struct = pd.read_parquet(STRUCT_PATH)

print("Shape:", df_struct.shape)
df_struct.head()

Shape: (15912393, 13)


,serialno,sporder,pwgtp,actor_class,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,income_tier_pct,mar_tier,commute_tier
0,2019GQ0000088,1,2,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,P0-20,Never_Married,NaN
1,2019GQ0000096,1,14,Adult,18-24,Female,White_NH,HS_or_less,Student,0-19k,P0-20,Never_Married,NaN
2,2019GQ0000153,1,4,Adult,18-24,Male,Black_NH,Some_college,Employed,0-19k,P0-20,Never_Married,Car
3,2019GQ0000198,1,17,Adult,65+,Male,White_NH,HS_or_less,Retired,20-49k,P20-40,Previously_Married,NaN
4,2019GQ0000205,1,11,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,<=0,<=0,Previously_Married,NaN


In [6]:
df_struct.info()

<class 'pandas.DataFrame'>
RangeIndex: 15912393 entries, 0 to 15912392
Data columns (total 13 columns):
 #   Column             Dtype   
---  ------             -----   
 0   serialno           str     
 1   sporder            int64   
 2   pwgtp              int64   
 3   actor_class        str     
 4   age_bin            category
 5   sex_label          str     
 6   race_eth           category
 7   edu_tier           str     
 8   emp_tier           str     
 9   income_tier_fixed  str     
 10  income_tier_pct    str     
 11  mar_tier           category
 12  commute_tier       category
dtypes: category(4), int64(2), str(7)
memory usage: 1.9 GB


In [7]:
# Sanity checks - Validate inputs (schema, missingness, and weights)

print("Rows, Cols:", df_struct.shape)
print("\nColumns:\n", df_struct.columns.tolist())

# Check critical columns exist (snake_case schema)
required_cols = ["pwgtp"]
missing_required = [c for c in required_cols if c not in df_struct.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Missingness snapshot
na_rate = (df_struct.isna().mean().sort_values(ascending=False) * 100).round(2)
print("\nTop missingness (%):")
print(na_rate.head(15))

# Weight health
w = df_struct["pwgtp"]
print("\npwgtp summary:")
print(w.describe())

bad_w = (~np.isfinite(w)) | (w <= 0)
print("\nBad pwgtp count:", int(bad_w.sum()))

if bad_w.any():
    print("\nExample bad pwgtp rows:")
    display(df_struct.loc[bad_w].head(10))

# Quick distribution checks (unweighted counts)
check_cols = [
    "actor_class", "age_bin", "sex_label", "race_eth", "edu_tier", "emp_tier",
    "income_tier_fixed", "income_tier_pct", "mar_tier", "commute_tier"
]
for col in check_cols:
    if col in df_struct.columns:
        print(f"\n{col} (top 10):")
        print(df_struct[col].value_counts(dropna=False).head(10))

Rows, Cols: (15912393, 13)

Columns:
 ['serialno', 'sporder', 'pwgtp', 'actor_class', 'age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed', 'income_tier_pct', 'mar_tier', 'commute_tier']

Top missingness (%):
commute_tier         54.56
income_tier_fixed    19.22
income_tier_pct      19.22
emp_tier             16.81
edu_tier              2.72
serialno              0.00
sporder               0.00
pwgtp                 0.00
actor_class           0.00
age_bin               0.00
sex_label             0.00
race_eth              0.00
mar_tier              0.00
dtype: float64

pwgtp summary:
count    1.591239e+07
mean     2.088860e+01
std      2.181002e+01
min      1.000000e+00
25%      9.000000e+00
50%      1.500000e+01
75%      2.400000e+01
max      1.071000e+03
Name: pwgtp, dtype: float64

Bad pwgtp count: 0

actor_class (top 10):
actor_class
Adult    12853377
Minor     3059016
Name: count, dtype: int64

age_bin (top 10):
age_bin
65+      3501248
55-64    2293256


In [8]:
# Step 0 — Configure balanced WOR sampling on 6 factors

TARGET_N = 1_000_000 

balance_cols = [
    "race_eth",
    "age_bin",
    "sex_label",
    "edu_tier",
    "income_tier_fixed",
    "mar_tier",
]

weight_col = "pwgtp"

# Guardrails: required columns
missing = [c for c in ([weight_col] + balance_cols) if c not in df_struct.columns]
if missing:
    raise ValueError(f"Missing required columns in df_struct: {missing}")

# Make Missing explicit + use string for stable grouping
for c in balance_cols:
    df_struct[c] = df_struct[c].astype("string").fillna("Missing")

# Basic weight checks
if (df_struct[weight_col] <= 0).any():
    raise ValueError("Found non-positive pwgtp values. Sampling requires pwgtp > 0.")

print("Ready. Rows:", f"{len(df_struct):,}", "| TARGET_N:", f"{TARGET_N:,}", "| Balancing cols:", balance_cols)

Ready. Rows: 15,912,393 | TARGET_N: 1,000,000 | Balancing cols: ['race_eth', 'age_bin', 'sex_label', 'edu_tier', 'income_tier_fixed', 'mar_tier']


In [9]:
# Step 1 — Compute 6-way cell allocations

TARGET_N = 1_000_000

# Compute cell totals
cell_stats = (
    df_struct
    .groupby(balance_cols, observed=True)
    .agg(
        cell_weight=(weight_col, "sum"),
        cell_count=(weight_col, "size")
    )
    .reset_index()
)

total_weight = cell_stats["cell_weight"].sum()

# Proportional allocation
cell_stats["n_alloc"] = (
    TARGET_N * (cell_stats["cell_weight"] / total_weight)
)

# Largest remainder method (no rounding bias)
cell_stats["n_floor"] = np.floor(cell_stats["n_alloc"]).astype(int)
remainder = TARGET_N - cell_stats["n_floor"].sum()

# Distribute remaining units to largest fractional parts
cell_stats["frac"] = cell_stats["n_alloc"] - cell_stats["n_floor"]
if remainder > 0:
    top_idx = cell_stats["frac"].nlargest(remainder).index
    cell_stats.loc[top_idx, "n_floor"] += 1

cell_stats["n_final"] = cell_stats["n_floor"]

# Safety cap (cannot sample more than available rows)
cell_stats["n_final"] = np.minimum(cell_stats["n_final"], cell_stats["cell_count"])

print("Total allocated rows:", cell_stats["n_final"].sum())
print("Number of 6-way cells:", len(cell_stats))
cell_stats.head()

Total allocated rows: 1000000
Number of 6-way cells: 4326


,race_eth,age_bin,sex_label,edu_tier,income_tier_fixed,mar_tier,cell_weight,cell_count,n_alloc,n_floor,frac,n_final
0,Asian_NH,0-5,Female,HS_or_less,Missing,Never_Married,288682,12339,868.510286,869,0.510286,869
1,Asian_NH,0-5,Female,Missing,Missing,Never_Married,260710,10711,784.355508,784,0.355508,784
2,Asian_NH,0-5,Male,HS_or_less,Missing,Never_Married,306310,12865,921.544764,922,0.544764,922
3,Asian_NH,0-5,Male,Missing,Missing,Never_Married,271708,11062,817.443390,817,0.443390,817
4,Asian_NH,13-17,Female,HS_or_less,Missing,Married,3879,201,11.670112,12,0.670112,12


In [ ]:
# Step 2 
# Reproducible weighted sampling driven by the fixed RNG initialized above

# Build fast lookup: 6-way cell -> n_final
n_map = cell_stats.set_index(balance_cols)["n_final"]

sampled_idx_parts = []
total_groups = 0
kept_groups = 0

for key, g in df_struct.groupby(balance_cols, observed=True, sort=False):
    total_groups += 1
    n = int(n_map.get(key, 0))
    if n <= 0:
        continue

    kept_groups += 1

    w = g[weight_col].to_numpy(dtype=float)
    if n >= len(g):
        # take all if allocation reaches/exceeds available rows
        sampled_idx_parts.append(g.index.to_numpy())
        continue

    # Efraimidis–Spirakis: sample without replacement with weights
    u = rng.random(len(g))
    keys = -np.log(u) / w  # smaller key = higher priority
    chosen_pos = np.argpartition(keys, n - 1)[:n]

    sampled_idx_parts.append(g.index.to_numpy()[chosen_pos])

sampled_idx = np.concatenate(sampled_idx_parts)
df_sample = df_struct.loc[sampled_idx].reset_index(drop=True)

print("Groups total:", total_groups, "| Groups sampled:", kept_groups)
print("Target rows:", f"{TARGET_N:,}")
print("Achieved rows:", f"{len(df_sample):,}")
print("Sample total pwgtp:", f"{df_sample[weight_col].sum():,}")
df_sample.head()

Groups total: 4326 | Groups sampled: 4124
Target rows: 1,000,000
Achieved rows: 1,000,000
Sample total pwgtp: 40,945,305


,serialno,sporder,pwgtp,actor_class,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,income_tier_pct,mar_tier,commute_tier
0,2023HU1043211,2,58,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car
1,2019HU1076190,2,46,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P20-40,Never_Married,Car
2,2019GQ0046130,1,12,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,P0-20,Never_Married,NaN
3,2019HU0403832,1,76,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Work_From_Home
4,2019HU0277198,1,64,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car


In [11]:
# Step 3 — Representativeness audit (1-way + optional 2-way)

req_cols = ["race_eth", "age_bin", "sex_label", "edu_tier", "income_tier_fixed", "mar_tier"]
wcol = "pwgtp"

def pop_pct_1way(df, col):
    s = df.groupby(col, observed=True)[wcol].sum()
    return (s / s.sum() * 100)

def samp_pct_1way(df, col):
    s = df[col].value_counts(normalize=True, dropna=False)
    return (s * 100)

def align_diff(a, b):
    idx = sorted(set(a.index) | set(b.index))
    a2 = a.reindex(idx, fill_value=0.0)
    b2 = b.reindex(idx, fill_value=0.0)
    d = (b2 - a2).abs()
    return float(d.max()), float(d.mean())

print("=" * 80)
print("AUDIT — 1-WAY MARGINS (Population=pwgtp-weighted full, Sample=row %)")
print("=" * 80)

results_1way = []
for col in req_cols:
    pop = pop_pct_1way(df_struct, col)
    samp = samp_pct_1way(df_sample, col)
    mx, mn = align_diff(pop, samp)
    results_1way.append((col, mx, mn))
    print(f"{col:<16} | max diff {mx:7.3f}% | mean diff {mn:7.3f}%")

# Optional: 2-way joints (pick a small set that matters)
joints = [
    ("race_eth", "age_bin"),
    ("race_eth", "sex_label"),
    ("edu_tier", "income_tier_fixed"),
    ("mar_tier", "income_tier_fixed"),
]

def pop_pct_2way(df, c1, c2):
    s = df.groupby([c1, c2], observed=True)[wcol].sum()
    return (s / s.sum() * 100)

def samp_pct_2way(df, c1, c2):
    s = df.groupby([c1, c2], observed=True).size()
    return (s / s.sum() * 100)

print("\n" + "=" * 80)
print("AUDIT — 2-WAY JOINTS (Population=pwgtp-weighted full, Sample=row %)")
print("=" * 80)

for c1, c2 in joints:
    pop = pop_pct_2way(df_struct, c1, c2)
    samp = samp_pct_2way(df_sample, c1, c2)
    mx, mn = align_diff(pop, samp)
    print(f"{c1}×{c2:<20} | max diff {mx:7.3f}% | mean diff {mn:7.3f}%")

AUDIT — 1-WAY MARGINS (Population=pwgtp-weighted full, Sample=row %)
race_eth         | max diff   0.002% | mean diff   0.001%
age_bin          | max diff   0.003% | mean diff   0.001%
sex_label        | max diff   0.001% | mean diff   0.001%
edu_tier         | max diff   0.002% | mean diff   0.001%
income_tier_fixed | max diff   0.001% | mean diff   0.001%
mar_tier         | max diff   0.002% | mean diff   0.001%

AUDIT — 2-WAY JOINTS (Population=pwgtp-weighted full, Sample=row %)
race_eth×age_bin              | max diff   0.001% | mean diff   0.000%
race_eth×sex_label            | max diff   0.001% | mean diff   0.000%
edu_tier×income_tier_fixed    | max diff   0.001% | mean diff   0.000%
mar_tier×income_tier_fixed    | max diff   0.001% | mean diff   0.000%


In [12]:
# Step 4 — Save modeling dataset

# Output file
OUTPUT_PATH = INTERIM_DIR / "acs_pums_5y"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_PATH / "us_structural_modeling_sample_v1.parquet"

# Save dataset
df_sample.to_parquet(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", f"{len(df_sample):,}")

Saved: /Users/marcomagnolo/Projects/us-audience-segmentation/data/interim/acs_pums_5y/us_structural_modeling_sample_v1.parquet
Rows: 1,000,000
